### Case 5: Multivariate X with gaussian marginals

With X of dimension d and all $X_i$ are gaussian marginals, solving one iteration of optimal bid and optimal allocation across Xi. The $X_i$ are generated from a normal distribution - means drawn from $\mathcal{N}(10, 4^2)$ and sigma from $\mathcal{U}(0.5,5)$.
Initital dependence structure is Gaussian Copula given by $\Sigma$.

1. Optimal Bid: Worst case VaR on the lower tail of the aggregate $Z = \sum_{i=1}^{d} X_i$ denoted by $\underline{VaR}_{\epsilon}(Z)$ where $\epsilon = 1 - \alpha = 0.1$. Solved using Rearrangement Algorithm. Additionally, solve the 
$VaR_{\epsilon}^{M}$ at $\rho = 1$ and  $VaR_{\epsilon}^{\Pi}$ at $\rho = 0$. 

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import rearrangement_algorithm as ra

In [2]:
rng = np.random.default_rng(0)
d = 20
mu = rng.normal(loc = 10, scale = 4, size = d)
sd = rng.uniform(0.5, 5, size = d)
a = np.ones(d)
alpha = 0.9; eps = 1 - alpha

# Marginal quantile functions for the weighted components a_i X_i
qf = [stats.norm(loc = a[j]*mu[j], scale = abs(a[j])*sd[j]).ppf for j in range(d)]

In [7]:
sd

array([0.62743852, 1.05927474, 3.51780987, 3.4123528 , 3.269233  ,
       2.22654899, 4.98744471, 4.91375902, 3.58493893, 3.42706674,
       3.59801029, 2.25014641, 1.10793427, 3.74669753, 2.86409445,
       1.89608844, 2.68625911, 4.50269525, 4.70319582, 2.11007839])

In [3]:
# P90 of each X_i: lower eps-quantile, before applying weights a
VaR_X = stats.norm.ppf(eps, loc = mu, scale = sd)
VaR_X

array([ 9.69882607,  8.11406534,  8.05343586,  6.04649439,  3.66763184,
        8.59294287,  8.8243326 ,  7.49108828,  2.59077496,  0.54635137,
        2.89586643,  7.28162527, -0.719998  ,  4.32324726,  1.34587148,
        4.64099547,  4.38038449,  2.96436322,  5.61913418, 11.46587922])

### Gaussian dependence and aggregate

$\Sigma$ is the copula correlation matrix; $E = \operatorname{diag}(\sigma)\Sigma\operatorname{diag}(\sigma)$ is the covariance matrix of $X$.
For $d > 1$, equicorrelation is positive semidefinite for $-1/(d-1) \leq \rho \leq 1$.
The copula generates uniforms $U$, then the marginal quantile functions give $X_i = F_i^{-1}(U_i)$.

In [ ]:
def corr_eq(d, rho):
    if d < 1: raise ValueError('d must be positive')
    rho_min = -1/(d - 1) if d > 1 else -1.
    if not np.isfinite(rho) or not rho_min <= rho <= 1:
        raise ValueError(f'rho must lie in [{rho_min:.4f}, 1] for d = {d}')
    return np.full((d, d), rho) + (1 - rho)*np.eye(d)


def u_gauss(Sigma, n, rng):
    # PSD matrices are allowed, including rho = 1 and the negative PSD limit
    Y = rng.multivariate_normal(np.zeros(len(Sigma)), Sigma, size = n, check_valid = 'raise')
    return stats.norm.cdf(Y)


# Swap u_gauss for another copula while keeping the marginals fixed
X_of = lambda U: stats.norm.ppf(U, loc = mu, scale = sd)
rho = 0.5; n = 100000
Sigma = corr_eq(d, rho)
E = np.outer(sd, sd) * Sigma
U = u_gauss(Sigma, n, rng)
X = X_of(U)
Z = X @ a  #sum of X_i when a = np.ones(d)

In [15]:
def VaR_gauss(rho):
    Sigma = corr_eq(d, rho)
    E = np.outer(sd, sd) * Sigma
    mu_Z = a @ mu; v_Z = a @ E @ a
    return mu_Z + stats.norm.ppf(eps) * np.sqrt(max(v_Z, 0))


VaR_M = VaR_gauss(1.)   #comonotonic: Sigma is all ones
VaR_Pi = VaR_gauss(0.)  #independent: Sigma = I, E is diagonal
VaR_rho = VaR_gauss(rho)
VaR_emp = np.quantile(Z, eps)

print(f'VaR^M  {VaR_M:.3f}; VaR^Pi  {VaR_Pi:.3f}')
print(f'VaR at rho = {rho:.2f}: exact {VaR_rho:.3f}  empirical {VaR_emp:.3f}')

VaR^M  107.823; VaR^Pi  166.601
VaR at rho = 0.50: exact 128.949  empirical 129.530


In [19]:
num_steps = 10000
worst_u, worst_o = ra.bounds_VaR(eps, qf, method = "lower", num_steps = num_steps, abstol = 1e-8)
VaR_worst_lo, VaR_worst_hi = worst_u[0], worst_o[0]

print(f'{VaR_worst_lo:.4f} <= underline VaR <= {VaR_worst_hi:.4f}')
print(f'_VaR_ {VaR_worst_lo:.4f} < VaR^M  {VaR_M:.3f}; VaR^Pi  {VaR_Pi:.3f}')

79.1764 <= underline VaR <= 79.1953
_VaR_ 79.1764 < VaR^M  107.823; VaR^Pi  166.601


Building the joint survival contour of X given by $S_X(p)=\mathbb{P}(X_i \ge p_i, \forall i \in {1\dots d})=\alpha$ where $p_i^{\alpha}=\mathcal{F}_{i}^-1(\epsilon)$ and check the combination of $p_i$s where the tangent has slope -1. 

The contour has joint survival $1-\epsilon=\alpha$; its coordinate asymptotes are $F_i^{-1}(\epsilon)$ as the other thresholds tend to $-\infty$.
We maximise $\sum_i p_i$ on this contour, using $u_i=F_i(p_i)$ with $0<u_i\le\epsilon$.
The start $u_i=\epsilon/d$ is feasible by the union bound. At a smooth interior optimum all partial derivatives of $S_X$ with respect to $p_i$ are equal (slope $-1$ in two dimensions).

For positive equicorrelation, a one-factor representation reduces the Gaussian CDF to deterministic one-dimensional quadrature, avoiding noisy finite differences during optimisation. Other correlation matrices use the multivariate Gaussian CDF and conditional Gaussian derivatives.

In [24]:
from scipy.integrate import quad
from scipy.optimize import minimize


def Phi_gauss(z, Sigma, tol = 1e-8):
    z = np.asarray(z, dtype = float); k = len(z)
    if k == 1: return stats.norm.cdf(z[0])
    if np.allclose(Sigma, np.eye(k)): return np.prod(stats.norm.cdf(z))
    if np.allclose(Sigma, np.ones((k, k))): return stats.norm.cdf(z.min())
    r = Sigma[0, 1]
    if 0 < r < 1 and np.allclose(Sigma, corr_eq(k, r)):
        f = lambda y: stats.norm.pdf(y) * np.prod(stats.norm.cdf((z - np.sqrt(r)*y)/np.sqrt(1 - r)))
        return quad(f, -np.inf, np.inf, epsabs = tol, epsrel = tol)[0]
    return stats.multivariate_normal.cdf(z, mean = np.zeros(k), cov = Sigma,
                                        allow_singular = True, maxpts = 1000000*k,
                                        abseps = tol, releps = tol)


def survival_gauss(p, mu, sd, Sigma, tol = 1e-8):
    return Phi_gauss((mu - np.asarray(p))/sd, Sigma, tol)


def survival_grad(p, mu, sd, Sigma):
    # dS/dp_i = -f_i(p_i) P(X_-i >= p_-i | X_i = p_i)
    z = (mu - np.asarray(p))/sd; k = len(z); grad = np.empty(k)
    for i in range(k):
        keep = np.arange(k) != i; r = Sigma[keep, i]
        C = Sigma[np.ix_(keep, keep)] - np.outer(r, r)
        s = np.sqrt(np.diag(C))
        cond = Phi_gauss((z[keep] - r*z[i])/s, C / np.outer(s, s)) if k > 1 else 1.
        grad[i] = -stats.norm.pdf(z[i])/sd[i] * cond
    return grad


def optimal_p(mu, sd, Sigma, eps = 0.1):
    mu = np.asarray(mu, dtype = float); 
    sd = np.asarray(sd, dtype = float); 
    k = len(mu)
    if not 0 < eps < 1: raise ValueError('eps must lie in (0, 1)')
    if np.any(sd <= 0): raise ValueError('sd must be positive')
    p_of = lambda u: stats.norm.ppf(u, loc = mu, scale = sd)
    # Exact comonotonic solution; the contour has corners here - same as pt of intersection of marginals
    if np.allclose(Sigma, np.ones((k, k))):
        return p_of(np.full(k, eps)), None
    dp_du = lambda u: sd / stats.norm.pdf(stats.norm.ppf(u))
    objective = lambda u: -p_of(u).sum()/sd.sum()
    jac = lambda u: -dp_du(u)/sd.sum()
    constraint = dict(type = 'ineq',
                      fun = lambda u: (survival_gauss(p_of(u), mu, sd, Sigma) - (1 - eps))/eps,
                      jac = lambda u: survival_grad(p_of(u), mu, sd, Sigma) * dp_du(u)/eps)
    res = minimize(objective, np.full(k, eps/k), jac = jac, method = 'SLSQP',
                   bounds = [(1e-12, eps)] * k, constraints = [constraint],
                   options = dict(ftol = 1e-9, maxiter = 500))
    if not res.success: raise RuntimeError(f'Allocation optimisation failed: {res.message}')
    p = p_of(res.x)
    if survival_gauss(p, mu, sd, Sigma, tol = 1e-10) < 1 - eps - 1e-6:
        raise RuntimeError('Allocation failed the joint survival check')
    return p, res

In [ ]:
p_star, res_p = optimal_p(mu, sd, Sigma, eps)
S_star = survival_gauss(p_star, mu, sd, Sigma, tol = 1e-10)

print(f'Joint survival = {S_star:.8f}  target = {1 - eps:.2f}')
print(f'Sum of p_i* = {p_star.sum():.4f} < _VaR_ {VaR_worst_lo:.4f} < VaR^M  {VaR_M:.3f}')
allocation_data = pd.DataFrame(dict(mu = mu, sd = sd, P90 = VaR_X, p_star = p_star))

Joint survival = 0.90000000  target = 0.90
Sum of p_i* = 45.1443 < _VaR_ 79.1764 < VaR^M  107.823


The sum of $p_i^*$ using Gaussian dependence $\Sigma$ is less than worst case $\underline{VaR}$.
Selecting the $\underline{VaR}$ as the optimal bid $b^*$ in this case. 

## Optimal bid allocation

1. Without redispatch: If there is no redispatch option, then what is the reliability level that the individual contribution given by $c_i$ will satisfy the bid $b^*$.
$$
\sum c_i = b^*; \qquad \mathbb{P}(X_i \ge c_i, \forall i \in {1\dots d})=\overline{\alpha}
$$


In [ ]:
def reliability_no_redispatch(c, mu, sd):
    # Sharp worst-case survival over all joint laws with these marginals
    z = (np.asarray(c) - mu)/sd
    failure = stats.norm.cdf(z)
    survival = stats.norm.sf(z)
    return dict(worst = max(0., 1 - failure.sum()), M = survival.min(), Pi = survival.prod())


b_star = VaR_worst_lo  #lower end of the RA bracket for the aggregate bid
# Allocation assumption: equal increments above p_star; replace c if another rule is used
c = p_star + (b_star - p_star.sum())/d
alpha_bar = reliability_no_redispatch(c, mu, sd)

print(f'Sum of c_i = {c.sum():.4f}  bid b* = {b_star:.4f}')
print(f"Without redispatch: worst = {alpha_bar['worst']:.6f}  rho = 1: {alpha_bar['M']:.6f}  rho = 0: {alpha_bar['Pi']:.6f}")
reliability_data = pd.DataFrame(dict(c = c, marginal_survival = stats.norm.sf((c - mu)/sd)))


Sum of c_i = 79.1764  bid b* = 79.1764
Without redispatch: worst = 0.000000  rho = 1: 0.535853  rho = 0: 0.205338


For a fixed allocation $c$, the three levels are
$$\overline{\alpha}_{\rm worst}=\max\{0,1-\sum_i F_i(c_i)\},\qquad
\overline{\alpha}_{M}=\min_i[1-F_i(c_i)],\qquad
\overline{\alpha}_{\Pi}=\prod_i[1-F_i(c_i)].$$
The first is a sharp pointwise bound over all dependence structures; no copula is needed. It does not identify the survival probability under the particular dependence producing worst-case aggregate VaR. For $d>2$, one dependence structure need not attain this lower bound at every allocation.

All three cases use the same bid and allocation for comparison. The equality $\sum_i c_i=b^*$ alone does not determine reliability: the allocation rule matters. The chosen equal increments illustrate one rule and do not optimise reliability. The RA bid is numerical, so `b_star` uses the lower end of its bracket.


2. With redispatch in single step: optimal allocation vector $c^*$